In [1]:
import pandas as pd
import os
import numpy as np
import ast
import collections
from pygenomeviz import GenomeViz
import ast
import pysam

In [2]:
myDF2=pd.read_csv("/LeeLab/HPRC/chromosomeY/Data/DAZ_Mappings/DAZNames_PSV.csv").set_index("Unnamed: 0")
myDF2['end']=[int(x.split("-")[1]) for x in myDF2.index]
myDF2['sample']=[x.split("_")[0] for x in myDF2['contig']]
goodDAZ = [x.split("_")[0] for x in set(myDF2['contig'])]

In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams.update({
    "pdf.fonttype": 42,  
    "ps.fonttype": 42,   
    "svg.fonttype": "none"  
})

mpl.rcParams["font.family"] = "Arial"

In [4]:
colorDict={'DAZ1_EXON1':'pink',
    'DAZ1_EXON2-7-12':'lightblue',
            'DAZ1_EXON3-8-13':'black',
            'DAZ1_EXON4-9-14':'red',
            'DAZ1_EXON5-10-15':'green',
            'DAZ1_EXON6-11-16':'purple',
           'DAZ1_EXON17':'yellow',
            'DAZ1_EXON18':'yellow',
           'DAZ1_EXON19':'yellow',
           'DAZ1_EXON20-22':'yellow',
           'DAZ1_EXON21':'yellow',
           'DAZ1_EXON23':'yellow',
           'DAZ1_EXON24':'yellow',
           'DAZ1_EXON25':'yellow',
           'DAZ1_EXON26':'lime',
           'DAZ1_EXON27':'grey',
           'DAZ1_EXON28':'blue',   
}

In [5]:
def add_feature(existing_features, new_feature):
    return existing_features + (new_feature,)

In [6]:
from typing import List, Optional
from Bio import Phylo

def tip_order_from_nexus_biopython(
    nexus_path: str,
    tree_label: Optional[str] = None
) -> List[str]:
    """
    Returns the left-to-right tip (sample) order from a NEXUS file using Biopython.
    If `tree_label` is given, picks that named tree; else the first tree in the file.
    """
    trees = list(Phylo.parse(nexus_path, "nexus"))
    if not trees:
        raise ValueError("No trees found in NEXUS file.")

    if tree_label is not None:
        matches = [t for t in trees if (t.name or "") == tree_label]
        if not matches:
            raise ValueError(f"Tree labeled '{tree_label}' not found.")
        tree = matches[0]
    else:
        tree = trees[0]

    tip_order = [clade.name for clade in tree.find_clades(order="preorder") if clade.is_terminal()]
    return tip_order

In [7]:
phylogeneticOrder=()
tips_bio = tip_order_from_nexus_biopython("/LeeLab/HPRC/chromosomeY/Information/phylogeny/142males_HGSVC_HPRC_CEPH_241125_150M-FOR_SHARING.nex")         # first tree

for sample in tips_bio:
    phylogeneticOrder = add_feature(phylogeneticOrder, sample)
   
print(phylogeneticOrder)
print(len(phylogeneticOrder))

('HG02984', 'HG01890', 'HG02647', 'HG02666', 'HG02668', 'HG03225', 'NA19043', 'NA19384', 'HG005', 'NA18952', 'NA18983', 'HG02572', 'HG03248', 'HG03098', 'HG03050', 'NA19239', 'HG01074', 'HG01109', 'HG01106', 'NA19331', 'HG01252', 'HG01457', 'HG03065', 'HG02717', 'HG03471', 'HG02011', 'HG03371', 'HG02486', 'HG02965', 'HG03139', 'NA19443', 'NA19317', 'NA19347', 'NA20346', 'HG02145', 'HG02258', 'HG02953', 'HG03130', 'HG03521', 'HG02554', 'NA18879', 'HG03209', 'NA18522', 'NA19700', 'HG02055', 'NA19705', 'NA18612', 'NA18620', 'HG04157', 'NA18989', 'NA18971', 'NA18974', 'HG02040', 'NA20870', 'HG03710', 'HG01099', 'HG03579', 'NA21093', 'HG04187', 'HG03009', 'HG03942', 'HG01167', 'HG00140', 'HG00321', 'NA20905', 'HG02492', 'HG02735', 'NA20805', 'NA20809', 'HG01255', 'HG01258', 'HG01433', 'HG003', 'HG002', 'HG03688', 'HG04199', 'HG01192', 'HG01530', 'HG03742', 'NA18608', 'NA18747', 'HG00280', 'HG00329', 'HG00290', 'HG00358', 'HG02015', 'HG02083', 'HG02514', 'HG02074', 'NA18534', 'HG02027', 'HG0

In [8]:
haplogroups={}
phylDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/HPRC_HGSVC3_sample_annotations_20March2025.txt",sep='\t')#
phylDF['GoodSampleNames']=[x.split("/")[1] if '/' in x else x for x in phylDF['sample']]
phylDF.set_index("GoodSampleNames", inplace=True)

In [9]:
for sample in phylogeneticOrder:
    if sample in phylDF.index:
        haplogroups[sample]=phylDF.at[sample,'haplogroup_ISOGG_v15.73']
    else:
        continue

haplogroups['NA12877']='R1b1a1b1a1a2e2'
haplogroups['NA12882']='R1b1a1b1a1a2e2'
haplogroups['NA12886']='R1b1a1b1a1a2e2'
haplogroups['200080']='R1b1a1b1a1a1c2b2b1a'
haplogroups['200084']='R1b1a1b1a1a1c2b2b1a'
haplogroups['200085']='R1b1a1b1a1a1c2b2b1a'

print(len(haplogroups))

140


In [10]:
#QC checked good AZFc regions
errDF=pd.read_csv("/LeeLab/HPRC/chromosomeY/QC/LocationFiles/AZFc_QC_File.csv").drop(columns=['Unnamed: 0'])

In [11]:
hgsvc = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/HGSVC3_chrYAmpliconicGene_Filtered_wLiftOffAnnotation_07162025.csv')
hprc = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/HPRC_chrYAmpliconicGene_Filtered_wLiftOffAnnotation_06162025.csv')
ceph = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/AmpliconicGenes/CEPH_chrYAmpliconicGene_Filtered_wLiftOffAnnotation_09092025.csv')


dazDFCEPH = ceph[ceph['geneName']=='DAZ1'].copy()
dazDFCEPH['Consortia']="CEPH"
dazDFCEPH['contigStart']=dazDFCEPH['contigStart'].astype(int)
dazDFCEPH['contigEnd']=dazDFCEPH['contigEnd'].astype(int)
dazDFCEPH.sort_values(by=['contig','contigStart'], inplace=True)

dazDFHGSVC = hgsvc[hgsvc['geneName']=='DAZ1'].copy()
dazDFHGSVC['Consortia']="HGSVC"
dazDFHGSVC['contigStart']=dazDFHGSVC['contigStart'].astype(int)
dazDFHGSVC['contigEnd']=dazDFHGSVC['contigEnd'].astype(int)
dazDFHGSVC.sort_values(by=['contig','contigStart'], inplace=True)

dazDFHPRC = hprc[hprc['geneName']=='DAZ1'].copy()
dazDFHPRC['Consortia']="HPRC"
dazDFHPRC['contigStart']=dazDFHPRC['contigStart'].astype(int)
dazDFHPRC['contigEnd']=dazDFHPRC['contigEnd'].astype(int)
dazDFHPRC.sort_values(by=['contig','contigStart'], inplace=True)

In [12]:
combinedDAZ0 = pd.concat([dazDFHPRC,dazDFHGSVC,dazDFCEPH])
combinedDAZ1 = combinedDAZ0[~combinedDAZ0['contig'].str.contains("random")].copy()
combinedDAZ = combinedDAZ1[combinedDAZ1['sampleName'].isin(set(errDF['sample']))].copy()

print(len(combinedDAZ1))
print(len(combinedDAZ))
combinedDAZ.reset_index(inplace=True)

505
382


In [13]:
assemblyDict={}
directory='/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/'
for file in os.listdir(directory):
    if '.fai' in file or '.gzi' in file or '.DS' in file:
        continue
    else:
        assemblyDict[file.split("_")[0]]= directory+file
print(len(assemblyDict))

143


In [14]:
good_samples = [s for s in phylogeneticOrder if s in goodDAZ]

sampleCombinations = [
    [a, b]
    for a, b in zip(good_samples, good_samples[1:])
]

In [15]:
len(sampleCombinations)

99

## Mine for Kmers

In [16]:
def revcomp(seq: str) -> str:
    comp = str.maketrans("ACGTNacgtn", "TGCANtgcan")
    return seq.translate(comp)[::-1]

In [17]:
def kmer_positions(sequence: str, k: int):
    """
    Return a dictionary mapping each k-mer to a list of (start, end) positions
    in the sequence.

    Positions are 0-based and end is exclusive, i.e. sequence[start:end] == kmer.
    """
    if k <= 0:
        raise ValueError("k must be a positive integer")
    if k > len(sequence):
        return {}

    seq = sequence.upper()
    pos_dict = {}

    for i in range(len(seq) - k + 1):
        kmer = seq[i:i + k]
        if kmer not in pos_dict:
            pos_dict[kmer] = []
        pos_dict[kmer].append((i + 1, i + k))

    return pos_dict


In [18]:
def generate_kmers(seq: str, k: int):
    if k <= 0:
        raise ValueError("k must be a positive integer.")
    if k > len(seq):
        return set()

    seq = seq.upper()
    revSeq = revcomp(seq)
    kmers = {seq[i:i+k] for i in range(len(seq) - k + 1)}
    revKmers = {revSeq[i:i+k] for i in range(len(revSeq) - k + 1)}

    uniqueKmers = {}
    for x in kmers:
        if x in uniqueKmers:
            continue
        else:
            uniqueKmers[x]=1

    for y in revKmers:
        if y in uniqueKmers:
            continue
        else:
            uniqueKmers[y]=1
            
    return uniqueKmers

In [19]:
sampleCoordinates={}
sampleCoordinatesnonUnique={}
sampleDAZDictUnique={}
sampleDAZDictNonUnique={}
sampleCoordinatesList=[]

for sample in phylogeneticOrder:
    if sample in goodDAZ:
        sampleCoordinates[sample] = {}
        sampleCoordinatesnonUnique[sample] = {}
        tempDict = {}

        tempDF = combinedDAZ[combinedDAZ['sampleName'] == sample].copy()

        for row in tempDF.index:
            geneCoordinate = (
                f"{tempDF.at[row,'contig']}:{tempDF.at[row,'contigStart']}-"
                f"{tempDF.at[row,'contigEnd']}"
            )
            sampleCoordinatesList.append(geneCoordinate)

            sequence = ''.join(
                pysam.faidx(assemblyDict[sample], geneCoordinate).split()[1:]
            )
            uniqueKmers = generate_kmers(sequence, 100)

            for kmer in uniqueKmers:
                if kmer in tempDict:
                    tempDict[kmer].append(geneCoordinate)
                else:
                    tempDict[kmer] = [geneCoordinate]

            sampleCoordinates[sample][geneCoordinate] = 0
            sampleCoordinatesnonUnique[sample][geneCoordinate] = 0

        tempDict2 = {}         
        nonUniqueTempDict = {}  

        for kmer, geneList in tempDict.items():
            if len(geneList)>=2:
                nonUniqueTempDict[kmer]=geneList
                for x in geneList:
                    sampleCoordinatesnonUnique[sample][x]+=1
            
            elif len(geneList)==1:
                tempDict2[kmer]=geneList[0]
                sampleCoordinates[sample][geneList[0]]+=1

            else:
                continue
                
        sampleDAZDictUnique[sample]=tempDict2
        sampleDAZDictNonUnique[sample]=nonUniqueTempDict
               

In [20]:
geneKmerLocation={}
geneCoordinateSequences={}
for sample in phylogeneticOrder:
    if sample in goodDAZ:

        tempDF = combinedDAZ[combinedDAZ['sampleName'] == sample].copy()

        for row in tempDF.index:

            orientation = tempDF.at[row,'orientation']
            geneCoordinate = (
                f"{tempDF.at[row,'contig']}:{tempDF.at[row,'contigStart']}-"
                f"{tempDF.at[row,'contigEnd']}"
            )

            sequence = ''.join(
                pysam.faidx(assemblyDict[sample], geneCoordinate).split()[1:]
            )
            positionDict = kmer_positions(sequence, 100)

            uniqueKmers=[]
            for kmer in sampleDAZDictUnique[sample]:
                if sampleDAZDictUnique[sample][kmer] == geneCoordinate:
                    uniqueKmers.append(kmer)
                else:
                    continue

            uniqueKmerPositions={}
            for kmer2 in uniqueKmers:
                try:
                    uniqueKmerPositions[kmer2]=positionDict[kmer2]
                except:
                    uniqueKmerPositions[kmer2]=positionDict[revcomp(kmer2)]
                
            geneKmerLocation[geneCoordinate]=uniqueKmerPositions
            geneCoordinateSequences[geneCoordinate]=sequence

In [21]:
from dna_features_viewer import GraphicFeature, GraphicRecord
from tqdm import tqdm

comparisonDict={}
for combo in tqdm(sampleCombinations):

    topSample=combo[0]
    bottomSample=combo[1]

    topGeneCoordinates = sampleCoordinates[topSample]
    bottomGeneCoordinates = sampleCoordinates[bottomSample]

    for gene in topGeneCoordinates:
        for gene2 in bottomGeneCoordinates:
            if gene in comparisonDict:
                comparisonDict[gene][gene2]={'Top':[], 'Bottom':[]}

            else:
                comparisonDict[gene]={gene2:{'Top':[], 'Bottom':[]}}
    
    for kmer in sampleDAZDictUnique[topSample]:
        if kmer in sampleDAZDictUnique[bottomSample]:
            topGene = sampleDAZDictUnique[topSample][kmer]
            bottomGene = sampleDAZDictUnique[bottomSample][kmer]

            for position in geneKmerLocation[bottomGene][kmer]:
                comparisonDict[topGene][bottomGene]['Bottom'].append(position)

            for position in geneKmerLocation[topGene][kmer]:
                comparisonDict[topGene][bottomGene]['Top'].append(position)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99/99 [00:00<00:00, 127.58it/s]


In [22]:
def merge_intervals_with_counts(intervals):
    """
    Merge overlapping intervals and count how many original intervals
    were merged into each.

    Parameters
    ----------
    intervals : list[tuple[int, int]]
        List of (start, end) tuples.

    Returns
    -------
    list[tuple[int, int, int]]
        List of (merged_start, merged_end, count_of_merged_intervals).
    """
    if not intervals:
        return []

    intervals = sorted(intervals, key=lambda x: x[0])

    merged = []

    cur_start, cur_end = intervals[0]
    cur_count = 1  

    for start, end in intervals[1:]:

        if start <= cur_end:
            cur_end = max(cur_end, end)
            cur_count += 1
        else:
            merged.append((cur_start, cur_end, cur_count))
            cur_start, cur_end = start, end
            cur_count = 1

    merged.append((cur_start, cur_end, cur_count))

    return merged

In [23]:
import ast
exonDict={}
for row in combinedDAZ.index:
    geneCoordinate = (
                f"{combinedDAZ.at[row,'contig']}:{combinedDAZ.at[row,'contigStart']}-"
                f"{combinedDAZ.at[row,'contigEnd']}"
            )

    exonDict[geneCoordinate]= ast.literal_eval(str(combinedDAZ.at[row,'Exons']))

In [24]:
colorDict={'DAZ1_EXON1':'pink',
    'DAZ1_EXON2-7-12':'lightblue',
            'DAZ1_EXON3-8-13':'black',
            'DAZ1_EXON4-9-14':'red',
            'DAZ1_EXON5-10-15':'green',
            'DAZ1_EXON6-11-16':'purple',
           'DAZ1_EXON17':'yellow',
            'DAZ1_EXON18':'yellow',
           'DAZ1_EXON19':'yellow',
           'DAZ1_EXON20-22':'yellow',
           'DAZ1_EXON21':'yellow',
           'DAZ1_EXON23':'yellow',
           'DAZ1_EXON24':'yellow',
           'DAZ1_EXON25':'yellow',
           'DAZ1_EXON26':'lime',
           'DAZ1_EXON27':'grey',
           'DAZ1_EXON28':'blue',   
}

In [25]:
letterExonDict={'17':'B', '18':'C','19':'D','20-22':'E','21':'F','23':'X','24':'Y','25':'Z'}

In [26]:
from dna_features_viewer import GraphicFeature, GraphicRecord
import matplotlib.pyplot as plt

#for firstGene in comparisonDict:
    for secondGene in comparisonDict[firstGene]:
        if len(comparisonDict[firstGene][secondGene]['Top'])>0 and len(comparisonDict[firstGene][secondGene]['Bottom'])>0:
            
            topSample    = firstGene
            bottomSample = secondGene
            
            topSampleSequence    = geneCoordinateSequences[topSample]
            bottomSampleSequence = geneCoordinateSequences[bottomSample]
            
            topSampleMin    = int(topSample.split(":")[1].split("-")[0])
            bottomSampleMin = int(bottomSample.split(":")[1].split("-")[0])
            
            top_features = []
            
            for start, end, count in merge_intervals_with_counts(
                    comparisonDict[topSample][bottomSample]['Top']):
                top_features.append(
                    GraphicFeature(
                        start=start,
                        end=end,
                        strand=0,
                        color='red',
                        label='KW:' + str(count)
                    )
                )
            
            exonList = exonDict[topSample]
            for exon in exonList:
                exonName = exon[9].split("EXON")[1]
            
                if exonName in letterExonDict.keys():
                    if str(exonName) == '17' and float(exon[1]) != 0.0:
                        label = 'A'
                    else:
                        label = letterExonDict[exonName]
                else:
                    label = None
            
                exon_start = (int(exon[5]) - topSampleMin) + 1
                exon_end   = (int(exon[6]) - topSampleMin) + 1
                strand     = +1 if exon[8] == '+' else -1
            
                if label:
                    top_features.append(
                        GraphicFeature(
                            start=exon_start,
                            end=exon_end,
                            strand=strand,
                            color=colorDict[exon[9]],
                            label=label
                        )
                    )
                else:
                    top_features.append(
                        GraphicFeature(
                            start=exon_start,
                            end=exon_end,
                            strand=strand,
                            color=colorDict[exon[9]]
                        )
                    )
            
            top_record = GraphicRecord(
                sequence_length=len(topSampleSequence),
                features=top_features
            )
            
            bottom_features = []
            
            for start, end, count in merge_intervals_with_counts(
                    comparisonDict[topSample][bottomSample]['Bottom']):
                bottom_features.append(
                    GraphicFeature(
                        start=start,
                        end=end,
                        strand=0,
                        color='red',
                        label='KW:' + str(count)
                    )
                )
            
            exonList = exonDict[bottomSample]
            for exon in exonList:
                exonName = exon[9].split("EXON")[1]
            
                if exonName in letterExonDict.keys():
                    if str(exonName) == '17' and float(exon[1]) != 0.0:
                        label = 'A'
                    else:
                        label = letterExonDict[exonName]
                else:
                    label = None
            
                exon_start = (int(exon[5]) - bottomSampleMin) + 1
                exon_end   = (int(exon[6]) - bottomSampleMin) + 1
                strand     = +1 if exon[8] == '+' else -1
            
                if label:
                    bottom_features.append(
                        GraphicFeature(
                            start=exon_start,
                            end=exon_end,
                            strand=strand,
                            color=colorDict[exon[9]],
                            label=label
                        )
                    )
                else:
                    bottom_features.append(
                        GraphicFeature(
                            start=exon_start,
                            end=exon_end,
                            strand=strand,
                            color=colorDict[exon[9]]
                        )
                    )
            
            bottom_record = GraphicRecord(
                sequence_length=len(bottomSampleSequence),
                features=bottom_features
            )
            
            fig, (ax_top, ax_bottom) = plt.subplots(
                2, 1, figsize=(15, 6), constrained_layout=True  # <-- avoids tight_layout warning
            )
            
            top_record.plot(ax=ax_top)
            bottom_record.plot(ax=ax_bottom)
            
            for ax in (ax_top, ax_bottom):
                for text in ax.texts:
                    # labels are letters ('A','B',...) so check against values
                    if text.get_text() in letterExonDict.values():
                        text.set_fontsize(10)
            
            ax_top.set_title(topSample+" ("+str(myDF2.at[topSample, 'MaxCount'])+")", fontsize=14, fontweight="bold",y=0.90)
            ax_bottom.set_title(bottomSample+" ("+str(myDF2.at[bottomSample, 'MaxCount'])+")", fontsize=14, fontweight="bold",y=0.90)
            #plt.show()
            #fig.savefig("/pairWiseKmers_100/"+str(topSample.split(":")[0])+"-"+str(myDF2.at[topSample, 'MaxCount'].replace("/","_"))+"_"+str(bottomSample.split(":")[0])+"-"+str(myDF2.at[bottomSample, 'MaxCount'].replace("/","_"))+".pdf", bbox_inches="tight")

IndentationError: unexpected indent (4173988531.py, line 5)